# PERPHECT Training with PBI-Scope Data

This notebook demonstrates how to train the PERPHECT phage-host interaction predictor using data streamed from PBI-Scope.

## Prerequisites

1. The PBI-Scope pipeline must have been run to build the database and sequence files
2. You must be connected to the analysis container via JupyterLab

### Connecting to JupyterLab

From your local machine, create an SSH tunnel to the analysis container:
```bash
ssh -L 8886:localhost:8888 <your-host>
```
Then open `http://localhost:8886` in your browser.

Inside the container, navigate to the `PERPHECT/` folder and open this notebook.

### Production Training

For long training runs, use the `train.py` script instead of this notebook:
```bash
docker compose run --rm analysis python /workspace/PERPHECT/train.py --epochs 20
```
See `README.md` for full documentation.

## 1. Setup & Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure PERPHECT directory is in path for local imports
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from pbi import quick_connect
from pbi.negative_examples import NegativeExampleGenerator
from pbi_adapter import PBIAdapter
from transforms import translate_sequence_onehot

## 1b. GPU Detection

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {len(gpus)} device(s)')
    for gpu in gpus:
        print(f'  - {gpu.name}')
    print('Training will use GPU (fast).')
else:
    print('No GPU detected. Training will use CPU (slow).')
    print('See README.md for GPU setup instructions.')

## 2. Connect to PBI-Scope Database

In [ ]:
retriever = quick_connect()
print(f"Connected to database.")
print(f"Has host data: {retriever._has_host_data}")

## 3. Explore Available Data

In [ ]:
# Get summary statistics
phage_meta = retriever.get_phage_metadata()
print(f"Total phages: {len(phage_meta)}")
print(f"\nPhage length distribution:")
print(phage_meta['Length'].describe())

In [ ]:
host_meta = retriever.get_host_metadata()
print(f"Total hosts: {len(host_meta)}")
print(f"\nHost genome length distribution:")
print(host_meta['Genome_Length'].describe())

In [ ]:
# Load all pairs from the database
all_pairs = retriever.get_phage_host_pairs(
    limit=1000,
    host_contig_mode='concat'
)
print(f"Loaded {len(all_pairs)} pairs from database")
all_pairs.head()

## 4. Classify Pairs & Generate Negatives

The database stores all phage-host associations, including both positive interactions (virulent, temperate) and negative interactions ("no interaction"). The `classify_pairs_by_interaction()` method queries the `private_interactions` table to determine the interaction type for each pair, then splits them into positives (label=1) and negatives (label=0).

We then generate additional synthetic negatives to reach the desired ratio.

In [ ]:
# Classify pairs by interaction type
positive_pairs, private_negatives = adapter.classify_pairs_by_interaction(all_pairs)
print(f"Positive pairs: {len(positive_pairs)}")
print(f"True negatives from private data: {len(private_negatives)}")

# Generate additional synthetic negatives
neg_gen = NegativeExampleGenerator(retriever)
generated_negatives = neg_gen.generate_random_negatives(
    positive_pairs,
    ratio=1.0
)
generated_negatives['negative_source'] = 'generated'
print(f"Generated synthetic negatives: {len(generated_negatives)}")

# Combine all negatives
import pandas as pd
if len(private_negatives) > 0 and len(generated_negatives) > 0:
    negative_pairs = pd.concat([private_negatives, generated_negatives], ignore_index=True)
elif len(private_negatives) > 0:
    negative_pairs = private_negatives
else:
    negative_pairs = generated_negatives

print(f"\nTotal negatives: {len(negative_pairs)}")
print(f"  - Private data: {len(private_negatives)}")
print(f"  - Generated: {len(generated_negatives)}")
negative_pairs.head()

## 5. Transform Data to PERPHECT Format

The `PBIAdapter` handles:
- Mapping PBI-Scope string IDs to integer IDs
- Zero-padding sequences to fixed lengths
- One-hot encoding DNA sequences
- Filtering sequences by minimum length

In [ ]:
adapter = PBIAdapter(
    retriever,
    bacterium_threshold=7_000_000,  # 7M bp (PERPHECT default)
    phage_threshold=200_000,        # 200K bp (PERPHECT default)
    bacterium_min_length=150_000,   # Filter smaller bacteria
    phage_min_length=1_500          # Filter smaller phages
)

couples_df, bacteria_df, phages_df = adapter.to_perphect_dataframes(
    positive_pairs,
    negative_pairs
)

print(f"Couples: {len(couples_df)}")
print(f"Bacteria: {len(bacteria_df)}")
print(f"Phages: {len(phages_df)}")
print(f"\nClass distribution:")
print(couples_df['interaction_type'].value_counts())

In [ ]:
# Preview the data
print("Couples (first 5):")
display(couples_df.head())

print(f"\nBacteria sequences shape: {bacteria_df['bacterium_sequence'].iloc[0].shape}")
print(f"Phage sequences shape: {phages_df['phage_sequence'].iloc[0].shape}")

## 6. Prepare Training Arrays

In [ ]:
from sklearn.model_selection import train_test_split

# Prepare integer-indexed couples and labels
couples, labels = adapter.prepare_training_data(
    positive_pairs,
    negative_pairs
)

print(f"Total pairs: {len(couples)}")
print(f"Positive: {int(labels.sum())}, Negative: {int(len(labels) - labels.sum())}")

# Train/validation/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    couples, labels, stratify=labels, test_size=0.3, shuffle=True, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_test, y_test, stratify=y_test, test_size=0.5, shuffle=True, random_state=42
)

print(f"\nTrain: {len(X_train)}, Valid: {len(X_valid)}, Test: {len(X_test)}")

## 7. Build PERPHECT Model

In [ ]:
import math
import tensorflow as tf
import keras
from keras.models import Model
from keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Concatenate

BACTERIUM_THRESHOLD = 7_000_000
PHAGE_THRESHOLD = 200_000

# Bacteria branch
input1 = Input(shape=(BACTERIUM_THRESHOLD, 4), name="bacterial_input")
conv1_1 = Conv1D(64, 30, strides=10, activation='relu', name='bacterial_conv_1')(input1)
maxpool1_1 = MaxPooling1D(15, strides=5, name='bacterial_maxpool_1')(conv1_1)
conv1_2 = Conv1D(32, 25, strides=10, activation='relu', name='bacterial_conv_2')(maxpool1_1)
maxpool1_2 = MaxPooling1D(10, strides=5, name='bacterial_maxpool_2')(conv1_2)
conv1_3 = Conv1D(32, 10, strides=5, activation='relu', name='bacterial_conv_3')(maxpool1_2)
maxpool1_3 = MaxPooling1D(2, strides=2, name='bacterial_maxpool_3')(conv1_3)
flatten_bact = Flatten(name='bacteria_features')(maxpool1_3)

# Phage branch
input2 = Input(shape=(PHAGE_THRESHOLD, 4), name="phage_input")
conv2_1 = Conv1D(64, 30, strides=10, activation='relu', name='phage_conv_1')(input2)
maxpool2_1 = MaxPooling1D(15, strides=5, name='phage_maxpool_1')(conv2_1)
conv2_2 = Conv1D(32, 25, strides=10, activation='relu', name='phage_conv_2')(maxpool2_1)
maxpool2_2 = MaxPooling1D(2, strides=2, name='phage_maxpool_2')(conv2_2)
flatten_phage = Flatten(name='phage_features')(maxpool2_2)

# Classification head
concat_features = Concatenate(name='concatenated_features')([flatten_bact, flatten_phage])
dense1 = Dense(100, activation='relu')(concat_features)
dropout1 = Dropout(0.10)(dense1)
dense2 = Dense(1, activation='sigmoid')(dropout1)

model = Model(name='Perphect', inputs=[input1, input2], outputs=dense2)

# Learning rate schedule
def step_decay(epoch):
    initial_lrate = 0.0004
    drop = 0.5
    epochs_drop = 4.0
    return initial_lrate * math.pow(drop, math.floor((1 + epoch) / epochs_drop))

optimizer = keras.optimizers.Adam()
model.compile(optimizer, 'binary_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Train the Model

We use the PBIAdapter's TensorFlow generator to feed data to the model. The generator handles:
- Fetching sequences from PBI-Scope on demand
- Zero-padding to fixed lengths
- One-hot encoding

In [ ]:
from sklearn.utils import class_weight

BATCH_SIZE = 16
EPOCHS = 3
STEPS_PER_EPOCH = 400

# Create generators
train_gen = adapter.create_tf_generator(
    X_train, y_train, batch_size=BATCH_SIZE, shuffle=True
)
valid_gen = adapter.create_tf_generator(
    X_valid, y_valid, batch_size=BATCH_SIZE, shuffle=False
)
valid_steps = math.ceil(len(X_valid) / BATCH_SIZE)

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    ),
    keras.callbacks.LearningRateScheduler(step_decay),
]

# Train (class_weight not supported with generators in Keras 3.x;
# data is already balanced via ratio=1.0 in negative generation)
history = model.fit(
    train_gen,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=EPOCHS,
    validation_data=valid_gen,
    validation_steps=valid_steps,
    callbacks=callbacks
)

## 9. Plot Training History

In [ ]:
from plotting_utils import historic_plots

acc_fig, loss_fig = historic_plots(history)
acc_fig.axes[0].set_title('Accuracy')
loss_fig.axes[0].set_title('Loss')

display(acc_fig)
display(loss_fig)

# Save figures
acc_fig.savefig('accuracy.png', dpi=150, bbox_inches='tight')
loss_fig.savefig('val_loss.png', dpi=150, bbox_inches='tight')
print('Saved accuracy.png and val_loss.png')

## 10. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Test predictions
test_gen = adapter.create_tf_generator(
    X_test, y_test, batch_size=BATCH_SIZE, shuffle=False
)
test_steps = math.ceil(len(X_test) / BATCH_SIZE)
test_predictions = model.predict(test_gen, steps=test_steps)
test_pred_labels = (test_predictions.flatten() > 0.5).astype(int)

# Classification report
print('Classification Report:')
print(classification_report(y_test, test_pred_labels, target_names=['Negative', 'Positive']))

# Confusion matrix
cm = confusion_matrix(y_test, test_pred_labels)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Negative', 'Positive'],
       yticklabels=['Negative', 'Positive'],
       title='Confusion Matrix',
       ylabel='True label', xlabel='Predicted label')
plt.colorbar(im, ax=ax)
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
import os

output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

# Save model
model.save(os.path.join(output_dir, 'model.h5'))

# Save test results
results_df = pd.DataFrame({
    'bacterium_id': X_test[:, 0],
    'phage_id': X_test[:, 1],
    'observations': y_test,
    'predictions': test_predictions.flatten(),
    'prediction_labels': test_pred_labels
})
results_df.to_csv(os.path.join(output_dir, 'results_test_set.csv'), index=False)

print(f"Model saved to {output_dir}/model.h5")
print(f"Results saved to {output_dir}/results_test_set.csv")

## 12. Scaling to Full Data

To train on all available data, remove the `limit` parameter when loading pairs:

```python
# Load ALL pairs (no limit)
all_pairs = retriever.get_phage_host_pairs(host_contig_mode='concat')
```

**Note:** This will load hundreds of thousands of pairs. For very large datasets, consider:
1. Increasing `batch_size` in the generator
2. Using `STEPS_PER_EPOCH = len(X_train) // BATCH_SIZE`
3. Training for more epochs with early stopping
4. The adapter caches sequences in memory, so ensure sufficient RAM

### Negative Source Tracking

The training tracks where each negative pair came from:
- **private_data**: True negatives from the database (interaction type = "no interaction")
- **generated**: Synthetic negatives created by `NegativeExampleGenerator`

This is logged during training and saved in `summary.json`.

### Using the Training Script

For production training runs, use `train.py` instead of this notebook:

```bash
# Quick test
docker compose run --rm analysis python /workspace/PERPHECT/train.py \
    --limit 1000 --epochs 3

# Full training with config file
docker compose run --rm analysis python /workspace/PERPHECT/train.py \
    --config /workspace/PERPHECT/config.yaml

# Custom run
docker compose run --rm analysis python /workspace/PERPHECT/train.py \
    --epochs 20 --batch-size 32 --output-dir /results/my_run
```

The script saves:
- `model_best.keras` and `model_final.keras`
- `training_log.csv` (epoch-by-epoch metrics)
- `accuracy.png` and `val_loss.png` plots
- `results_test_set.csv` (predictions on test set)
- `summary.json` (run metadata including negative source proportions)
- `config.json` (parameters used)